In [1]:

import ftplib
import ftputil
import ftputil.session

def custom_active_session_factory(host, user, password):
    """
    A custom session factory that connects in Active mode (passive disabled).
    """
    ftp = ftplib.FTP()
    ftp.connect(host, 21)
    ftp.login(user, password)
    # Disable passive ⇒ enable Active mode:
    ftp.set_pasv(False)
    return ftp

def download_folder_recursive(
    ftp_host, 
    remote_dir, 
    local_dir, 
    debug=False
):
    """
    Recursively download the entire directory tree from `remote_dir` to `local_dir` using ftputil.
    1) Creates `local_dir` if it doesn't exist.
    2) For each item in `remote_dir`, downloads if it's a file.
    3) If it's a subdirectory, calls itself recursively.

    :param ftp_host:   An ftputil.FTPHost instance.
    :param remote_dir: The remote directory to download from (string).
    :param local_dir:  The local directory to mirror to (string).
    :param debug:      If True, prints debug messages.
    """
    # Create local_dir if not existing
    os.makedirs(local_dir, exist_ok=True)
    
    # Get listing of items in remote_dir
    items = ftp_host.listdir(remote_dir)
    
    for item in items:
        remote_path = ftp_host.path.join(remote_dir, item)
        local_path  = os.path.join(local_dir, item)

        # Check if it's a directory
        if ftp_host.path.isdir(remote_path):
            # Recurse
            if debug:
                print(f"[DIR]  {remote_path} → {local_path}")
            download_folder_recursive(ftp_host, remote_path, local_path, debug=debug)
        elif ftp_host.path.isfile(remote_path):
            # Download file
            if debug:
                print(f"[FILE] {remote_path} → {local_path}")
            ftp_host.download(remote_path, local_path)
        else:
            # Possibly a symlink or special file
            if debug:
                print(f"[SKIP] {remote_path} (not a file or directory)")


In [2]:
#!/usr/bin/env python
# coding: utf-8

import os
import paramiko
import time
import threading
from scp import SCPClient
from datetime import datetime
import ftplib  # for FTP transfers

class WiFiRouter:
    def __init__(
        self, 
        router_ip, 
        router_username, 
        router_password,
        key_filename=None,
        debug=False,
        ftp_host=None,
        ftp_user=None,
        ftp_password=None
    ):
        """
        Initialize the WiFiRouter object and create an SSH connection to the router.

        :param router_ip:       The IP address of the router.
        :param router_username: The username to login to the router.
        :param router_password: The password to login to the router.
        :param key_filename:    (Optional) Path to an SSH key file if password is not used.
        :param debug:           If True, prints additional debug info.
        :param ftp_host:        (Optional) Hostname/IP for FTP transfers.
        :param ftp_user:        (Optional) Username for FTP transfers.
        :param ftp_password:    (Optional) Password for FTP transfers.
        """
        self.router_ip = router_ip
        self.router_username = router_username
        self.router_password = router_password
        self.key_filename = key_filename
        self.debug = debug

        # FTP credentials (if needed)
        self.ftp_host = ftp_host
        self.ftp_user = ftp_user
        self.ftp_password = ftp_password

        # Internal state
        self.ssh = None
        self.sending = False
        self.interface = None
        self.capture_thread = None  # Thread for CSI capturing

        # Initialize SSH
        self._init_ssh_connection()

    def _init_ssh_connection(self):
        """Sets up and opens the SSH connection."""
        if self.debug:
            print(f"Initializing SSH connection to {self.router_ip}...")

        self.ssh = paramiko.SSHClient()
        self.ssh.load_system_host_keys()
        self.ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

        self.ssh.connect(
            self.router_ip,
            username=self.router_username,
            password=self.router_password,
            key_filename=self.key_filename
        )
        
        if self.debug:
            print("SSH connection established.")

    def send_command(self, cmd, sleep=0, get_pty=False):
        """
        Sends a command over SSH and returns the output.

        :param cmd:     Command to be executed.
        :param sleep:   Time (in seconds) to sleep after executing the command.
        :param get_pty: Whether to request a pseudo-tty.
        :return:        List of lines of the command output.
        """
        if self.debug:
            print(f"Executing command: {cmd}")

        ssh_stdin, ssh_stdout, ssh_stderr = self.ssh.exec_command(cmd, get_pty=get_pty)
        output = ssh_stdout.readlines()
        time.sleep(sleep)
        return output
    
    def transfer_folder_via_ftp_recursive(
            self, 
            remote_folder, 
            local_folder
        ):
            """
            Recursively download an entire folder tree from remote_folder to local_folder
            on the router's FTP, using ACTIVE mode. Subdirectories included.
            """
            remote_folder = remote_folder[4:]
            # 1. Ensure we have FTP credentials
            if not all([self.ftp_host, self.ftp_user, self.ftp_password]):
                raise RuntimeError("FTP credentials or host missing. Cannot transfer via FTP.")
    
            # 2. Create an FTPHost with active mode
            if self.debug:
                print(f"Connecting via ftputil (active mode) to {self.ftp_host}...")
    
            with ftputil.FTPHost(
                self.ftp_host,
                self.ftp_user,
                self.ftp_password,
                session_factory=custom_active_session_factory
            ) as ftp_host:
                if self.debug:
                    print(f"Connected. Now downloading recursively from '{remote_folder}' to '{local_folder}'...")
                
                # 3. Call our recursive function
                download_folder_recursive(ftp_host, remote_folder, local_folder, debug=self.debug)
    

    def transfer_file(self, remote_path, local_path, use_ftp=False):
        """
        Transfers a file from the router to the local machine.
        
        :param remote_path: Path to the file on the router.
        :param local_path:  Path on local machine where file will be saved.
        :param use_ftp:     If True, use FTP for transfer; if False, use SCP.
        """
        if use_ftp:
            if self.debug:
                print(f"Transferring via FTP from {remote_path} to {local_path}...")
            if not all([self.ftp_host, self.ftp_user, self.ftp_password]):
                raise RuntimeError("FTP credentials or host missing. Cannot transfer via FTP.")

            try:
                with ftplib.FTP(self.ftp_host) as ftp:
                    ftp.login(self.ftp_user, self.ftp_password)
                    ftp.set_pasv(False)
                    print(remote_path)
                    print(local_path)
                    with open(local_path, 'wb') as f:
                        ftp.retrbinary(f"RETR {remote_path}", f.write)
            except ftplib.all_errors as e:
                raise RuntimeError(f"FTP transfer failed: {e}")

            if self.debug:
                print(f"File transfer (FTP) complete: {remote_path} -> {local_path}")
        else:
            # Use SCP
            if self.debug:
                print(f"Transferring via SCP from {remote_path} to {local_path}...")

            with SCPClient(self.ssh.get_transport()) as scp:
                scp.get("/mnt/"+remote_path, local_path)

            if self.debug:
                print(f"File transfer (SCP) complete: {remote_path} -> {local_path}")

    def setup_router(self):
        """
        Load modules, set system date, etc.
        """
        current_time = datetime.today().strftime("%Y-%m-%d %H:%M:%S")
        self.send_command(f"date -s '{current_time}'", sleep=1)

        # Install Firmware
        self.send_command("cp /mnt/*/jffs/* /jffs/", sleep=1)
        # Commands to unload/load modules
        self.send_command("cd /jffs; /usr/sbin/wl -i eth5 monitor 0", sleep=1)
        self.send_command("cd /jffs; /usr/sbin/wl -i eth6 monitor 0", sleep=1)
        self.send_command("cd /jffs; ifconfig eth5 down", sleep=1)
        self.send_command("cd /jffs; ifconfig eth6 down", sleep=1)
        self.send_command("cd /jffs; /usr/sbin/wl -i eth5 down", sleep=1)
        self.send_command("cd /jffs; /usr/sbin/wl -i eth6 down", sleep=1)

        self.send_command("cd /jffs; /sbin/rmmod dhd", sleep=1)
        self.send_command("cd /jffs; /sbin/insmod dhd.ko", sleep=1)

    def set_power(self, value = 100, reboot=False): # MAX is 100
        self.send_command("nvram set wl0_txpower={}".format(value), sleep=1)
        self.send_command("nvram set wl1_txpower={}".format(value), sleep=1)
        self.send_command("nvram commit".format(value), sleep=1)
        if reboot:
            self.send_command("reboot".format(value), sleep=3)

    def configure_monitor_mode(self, channel="6/20", core=15, mac_addresses=None):
        """
        Configure the router for monitor mode, picking the interface (eth5 or eth6)
        based on the channel (2.4 vs 5 GHz).
        
        :param channel:       e.g. "6/20" or "149/20"
        :param core:          MCP core setting (e.g., 15)
        :param mac_addresses: A single MAC or list of MACs; if None, collects all possible MACs.
        :return:             MCP code (string)
        """
        # Parse channel
        freq_str = channel.split('/')[0]
        try:
            freq = int(freq_str)
        except ValueError:
            freq = 6  # default to 2.4 GHz if parsing fails

        # Choose interface
        if freq < 30:
            self.interface = "eth5"  # 2.4 GHz
        else:
            self.interface = "eth6"  # 5 GHz

        if self.debug:
            print(f"Channel: {channel}, freq={freq} → Using interface: {self.interface}")

        # Bring up interface
        self.send_command(f"cd /jffs; /usr/sbin/wl -i {self.interface} up", sleep=1)
        self.send_command(f"cd /jffs; /sbin/ifconfig {self.interface} up", sleep=1)

        # Build MCP command
        mac_args = ""
        if mac_addresses:
            if isinstance(mac_addresses, str):
                mac_args = f" -m {mac_addresses}"
            elif isinstance(mac_addresses, list):
                for mac in mac_addresses:
                    mac_args += f" -m {mac}"

        cmd = f"cd /jffs; ./mcp -c {channel} -C {core} -N 1{mac_args}"
        if self.debug:
            print(f"MCP command: {cmd}")

        output = self.send_command(cmd)
        mcp_code = output[0].strip() if output else ""

        # Insert MCP code into nexutil
        if mcp_code:
            cmd_nexutil = f"cd /jffs; ./nexutil -I{self.interface} -s500 -b -l34 -v{mcp_code}"
            self.send_command(cmd_nexutil, sleep=1)
            # Turn on monitor mode
            cmd_monitor = f"cd /jffs; /usr/sbin/wl -i {self.interface} monitor 1"
            self.send_command(cmd_monitor, sleep=1)
        
        return mcp_code

    def _capture_csi(
        self,
        duration,
        remote_directory,
        exp_name,
        delete_prev_pcap
    ):
        """
        Internal method that performs CSI capture in a background thread.
        Runs tcpdump in background, sleeps, then kills tcpdump automatically.
        """
        self.sending = True

        try:
            transport = self.ssh.get_transport() if self.ssh else None
            if not transport or not transport.is_active():
                raise RuntimeError("SSH connection is not active.")

            if not self.interface:
                raise RuntimeError(
                    "Interface not configured. Call configure_monitor_mode() first."
                )

            # 1) Ensure the remote directory exists
            mkdir_cmd = f"mkdir -p {remote_directory}"
            self.send_command(mkdir_cmd, sleep=0)

            # 2) Optionally delete old .pcap files
            if delete_prev_pcap:
                clear_cmd = f"rm -f {remote_directory}/*.pcap"
                self.send_command(clear_cmd, sleep=1)

            # 3) Build the filename
            date_str = datetime.today().strftime("%Y-%m-%d_%H-%M-%S")
            filename = f"{exp_name}_{date_str}.pcap"
            full_remote_path = f"{remote_directory}/{filename}"

            # 4) Construct tcpdump command, running in background on the router
            tcpdump_cmd = (
                f"nohup /jffs/tcpdump -i {self.interface} dst port 5500 "
                f"-w {full_remote_path} > /dev/null 2>&1 &"
            )

            if self.debug:
                print(
                    f"Starting CSI capture on {self.interface}...\n"
                    f"Remote pcap file: {full_remote_path}\n"
                    f"Capture duration: {duration} seconds\n"
                    f"delete_prev_pcap={delete_prev_pcap}"
                )

            # 5) Start tcpdump
            self.send_command(tcpdump_cmd, sleep=2)

            # 6) Sleep for 'duration' seconds
            time.sleep(duration)

            # 7) Kill tcpdump
            self.send_command("killall tcpdump", sleep=1)
            if self.debug:
                print("tcpdump killed automatically after duration.")

            return full_remote_path
        except (EOFError, paramiko.SSHException, RuntimeError) as exc:
            if self.debug:
                print(f"CSI capture aborted: {exc}")
            return None
        finally:
            self.sending = False

    
    def read_packets_count(self, full_remote_path=None, MACs = None):
        def count_elements(lst):
            """
            Count the number of occurrences of each element in a list.
            
            Parameters:
                lst (list): The input list.
                
            Returns:
                dict: A dictionary with elements as keys and their counts as values.
            """
            lst = [item.strip() for item in lst]
            element_count = {}
            for element in lst:
                if element in element_count:
                    element_count[element] += 1
                else:
                    element_count[element] = 1
            return element_count

        tcpdump_cmd_read = (
            f"nohup /jffs/tcpdump -nn -e -r {full_remote_path}" + "| awk '{print $2}' | sort"
        )
        captured_packets = self.send_command(tcpdump_cmd_read, sleep=0)
        counts = count_elements(captured_packets)
        if self.debug:
            # print(full_remote_path)
            if MACs:
                for mac_adr in MACs:
                    if mac_adr in counts:
                        print(counts[mac_adr])
                    else:
                        print(mac_adr, ": 0")
            else:
                print(counts)
            print("=======================")
        return counts
        
    def start_csi_capture_in_thread(
        self, 
        duration=10, 
        remote_directory="/mnt/CSI_USB", 
        exp_name="Test_CSI_1_",
        delete_prev_pcap=False
    ):
        """
        Launches a new thread for CSI capture (tcpdump). 
        After 'duration' seconds, tcpdump is automatically killed.

        :param duration:         How many seconds to capture.
        :param remote_directory: Directory on the router to store .pcap files.
        :param exp_name:         Experiment name to include in the filename.
        :param delete_prev_pcap: If True, remove old .pcap files from remote_directory before capturing.
        """
        if self.debug:
            print("Spawning CSI capture thread...")

        if self.capture_thread and self.capture_thread.is_alive():
            raise RuntimeError("A capture thread is already running. Stop it before starting a new one.")

        self.capture_thread = threading.Thread(
            target=self._capture_csi, 
            args=(duration, remote_directory, exp_name, delete_prev_pcap), 
            daemon=True
        )
        self.capture_thread.start()

    def stop_csi_capture(self):
        """
        Allows manual stopping of the capture before 'duration' has elapsed.
        Kills tcpdump if it's still running.
        """
        if self.sending:
            if self.debug:
                print("Manually stopping CSI capture...")
            self.send_command("killall tcpdump", sleep=1)
            self.sending = False

    def retrieve_files(
        self, 
        local_directory="/Users/YourUsername/Documents", 
        remote_directory="/mnt/CSI_USB", 
        use_ftp=False
    ):
        """
        Retrieves all pcap files from the router's remote_directory to a local directory.

        :param local_directory:  Local path to save the .pcap files.
        :param remote_directory: Directory on the router side to look for .pcap files.
        :param use_ftp:          If True, use FTP; if False, use SCP.
        """
        # 1) Create local directory if it doesn't exist
        os.makedirs(local_directory, exist_ok=True)
        
        # 2) List all pcap files in remote_directory
        all_files = self.send_command(f"ls {remote_directory}")
        pcap_files = [f.strip() for f in all_files if f.endswith(".pcap\n")]
        
        # 3) Download each .pcap file
        for pcap_file in pcap_files:
            remote_path = f"{remote_directory}/{pcap_file}"
            local_path = os.path.join(local_directory, pcap_file)
            self.transfer_file(remote_path[4:], local_path, use_ftp=use_ftp)

    def close(self):
        """Close the SSH connection."""
        if self.ssh:
            self.ssh.close()
        if self.debug:
            print("SSH connection closed.")



In [ ]:
# --------------- Example usage ---------------
# Replace with your actual credentials/paths
router_ip       = "192.168.50.1"
router_username = "admin"
router_password = "wirlabwirlab"
key_filename    = "/Users/navidhasanzadeh/Downloads/id_rsa.pub"
channel_bw = " 157/80"

# (Optional) FTP server details
ftp_host        = router_ip
ftp_user        = router_username
ftp_password    = router_password

# Initialize
router = WiFiRouter(
    router_ip=router_ip,
    router_username=router_username,
    router_password=router_password,
    key_filename=key_filename,
    ftp_host=ftp_host,
    ftp_user=ftp_user,
    ftp_password=ftp_password,
    debug=True
)

# 1. Setup router
router.setup_router()
# 2. Configure monitor mode
router.configure_monitor_mode(channel=channel_bw, core=15, mac_addresses=['34:7D:E4:4F:EB:FD'])
#    e.g., mac_addresses="2C:CF:67:32:07:78"

Initializing SSH connection to 192.168.50.1...
SSH connection established.
Executing command: date -s '2025-11-16 19:14:50'
Executing command: cp /mnt/*/jffs/* /jffs/
Executing command: cd /jffs; /usr/sbin/wl -i eth6 monitor 0
Executing command: cd /jffs; ifconfig eth6 down
Executing command: cd /jffs; /usr/sbin/wl -i eth6 down
Executing command: cd /jffs; /sbin/rmmod dhd; /sbin/insmod dhd.ko


In [ ]:
save_directory = "/mnt/CSI_USB/"
filename = "Test"
duration = 2 * 60

In [45]:
# 3. Start capturing in a separate thread
router.start_csi_capture_in_thread(
    duration=duration, 
    remote_directory=save_directory,
    exp_name=filename,
    delete_prev_pcap=False
)

# # Main thread can do other tasks
# for i in range(3):
#     print(f"[Main Thread] Doing other tasks... {i}")
#     time.sleep(5)
# # (Optional) Stop capturing early:
# # router.stop_csi_capture()

# Wait for the capture thread to finish
if router.capture_thread:
    router.capture_thread.join()

Spawning CSI capture thread...
Executing command: mkdir -p /mnt/CSI_USB/
Starting CSI capture on eth6...
Remote pcap file: /mnt/CSI_USB//Test_2025-11-12_20-58-11.pcap
Capture duration: 120 seconds
delete_prev_pcap=False
Executing command: nohup /jffs/tcpdump -i eth6 dst port 5500 -w /mnt/CSI_USB//Test_2025-11-12_20-58-11.pcap > /dev/null 2>&1 &
Executing command: killall tcpdump
tcpdump killed automatically after duration.


In [46]:
pcap_files = router.send_command(f"ls {save_directory}/*.pcap", sleep=0)
for pcap_file in pcap_files:
    counts = router.read_packets_count(pcap_file.strip(), MACs = None)

Executing command: ls /mnt/CSI_USB//*.pcap
Executing command: nohup /jffs/tcpdump -nn -e -r /mnt/CSI_USB//Test_2025-11-12_20-39-37.pcap| awk '{print $2}' | sort
{'4e:45:58:4d:4f:4e': 224}
Executing command: nohup /jffs/tcpdump -nn -e -r /mnt/CSI_USB//Test_2025-11-12_20-58-11.pcap| awk '{print $2}' | sort
{'4e:45:58:4d:4f:4e': 47766}


In [47]:
# 4. Transfer the .pcap files to your local machine
#    Switch `use_ftp=True` if you want to retrieve via FTP
router.retrieve_files(
    local_directory="/Users/navidhasanzadeh/Documents",
    remote_directory=save_directory,
    use_ftp=False
)


Executing command: ls /mnt/CSI_USB/
Transferring via SCP from /CSI_USB//Test_2025-11-12_20-39-37.pcap to /Users/navidhasanzadeh/Documents/Test_2025-11-12_20-39-37.pcap...
File transfer (SCP) complete: /CSI_USB//Test_2025-11-12_20-39-37.pcap -> /Users/navidhasanzadeh/Documents/Test_2025-11-12_20-39-37.pcap
Transferring via SCP from /CSI_USB//Test_2025-11-12_20-58-11.pcap to /Users/navidhasanzadeh/Documents/Test_2025-11-12_20-58-11.pcap...
File transfer (SCP) complete: /CSI_USB//Test_2025-11-12_20-58-11.pcap -> /Users/navidhasanzadeh/Documents/Test_2025-11-12_20-58-11.pcap


In [ ]:
PC_store_folder = "/Users/navidhasanzadeh/Documents/my_files"
remote_folder_on_router_flash_drive = "/mnt/sda1/data"
router.transfer_folder_via_ftp_recursive(remote_folder_on_router_flash_drive, PC_store_folder)

In [ ]:
# 5. Close SSH
router.close()
